# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imtiyazsoomro/flyrank-ml-internship-imtiyaz/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a Random Forest Classifier. It is an ensemble method that handles non-linear relationships—like the complex interaction between a page's age and its impression volume—exceptionally well without requiring extensive feature scaling. It also provides straightforward feature importance metrics out of the box, which is critical for explaining the model's reasoning to content teams during the playbook phase.

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Setup environment and load data
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/imtiyazsoomro/flyrank-ml-internship-imtiyaz.git
    %cd flyrank-ml-internship-imtiyaz

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Prepare features and target
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
feature_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']

# Handle missing values by imputing the median
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['is_declining']

print("Random Forest chosen and dependencies loaded.")

Random Forest chosen and dependencies loaded.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

To ensure an honest evaluation, I am using a standard 80/20 randomized split. While a time-aware split is often ideal for time-series data, our current dataset is a single cross-sectional snapshot (using aggregated 90-day windows). Therefore, stratifying the split by the target label (is_declining) ensures both the training and testing sets maintain the exact same proportion of declining pages, preventing class imbalance from skewing the model's evaluation.

In [6]:
# Create an 80/20 train-test split, stratified by the target label
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train):,} rows")
print(f"Testing set size: {len(X_test):,} rows")
print(f"Target distribution in test set: {y_test.mean():.1%} declining")

Training set size: 24,000 rows
Testing set size: 6,000 rows
Target distribution in test set: 54.2% declining


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

This section trains the Random Forest on the training split and compares its predictive performance against the Week 4 hardcoded baseline (days_since_last_update > 180) on the exact same testing split.

In [7]:
# 1. Evaluate the Week 4 Baseline on the test set
# Baseline Rule: Predict decline (1) if days_since_last_update > 180
baseline_preds = (X_test['days_since_last_update'] > 180).astype(int)

baseline_acc = accuracy_score(y_test, baseline_preds)
baseline_prec = precision_score(y_test, baseline_preds)

# 2. Train the Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# 3. Evaluate the Model on the test set
model_preds = rf_model.predict(X_test)
model_acc = accuracy_score(y_test, model_preds)
model_prec = precision_score(y_test, model_preds)

# 4. Print Comparison Table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision'],
    'Week 4 Baseline': [baseline_acc, baseline_prec],
    'Random Forest': [model_acc, model_prec]
})

print("=== MODEL VS BASELINE COMPARISON ===")
print(comparison_df.to_string(index=False))

=== MODEL VS BASELINE COMPARISON ===
   Metric  Week 4 Baseline  Random Forest
 Accuracy         0.456667        0.68600
Precision         0.394737        0.67449


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest outperforms the hardcoded rule, but it is not perfect. By extracting feature importances, we observe the model leans heavily on impressions_90d and avg_position.

A review of the confusion matrix shows that our False Positives (predicting a decline when the page is stable) often occur on high-impression pages. These pages naturally experience high traffic variance, tricking the model into interpreting standard seasonal fluctuations as an actionable decline.

In [8]:
# Extract and display feature importances
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("=== FEATURE IMPORTANCES ===")
print(importances.to_string(index=False))

# Display the confusion matrix for error analysis
cm = confusion_matrix(y_test, model_preds)
print("\n=== CONFUSION MATRIX ===")
print(f"True Negatives (Stable predicted Stable): {cm[0][0]}")
print(f"False Positives (Stable predicted Decline): {cm[0][1]} <-- Variance trap")
print(f"False Negatives (Decline predicted Stable): {cm[1][0]}")
print(f"True Positives (Decline predicted Decline): {cm[1][1]}")

=== FEATURE IMPORTANCES ===
               Feature  Importance
       impressions_90d    0.344013
      content_age_days    0.234303
          avg_position    0.228328
            word_count    0.128083
days_since_last_update    0.065273

=== CONFUSION MATRIX ===
True Negatives (Stable predicted Stable): 1472
False Positives (Stable predicted Decline): 1276 <-- Variance trap
False Negatives (Decline predicted Stable): 608
True Positives (Decline predicted Decline): 2644


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.